1.


Introduction
Large Language Models (LLMs) have transformed the landscape of natural language processing. However, the immense scale of these models, often containing billions or even trillions of parameters, introduces significant challenges in terms of inference efficiency. This lesson focuses on innovative techniques aimed at making LLMs smaller and enhancing the speed of token generation.

The following strategies will be explored:

Quantization: Reducing numerical precision to decrease model size and improve speed.
Pruning: Identifying and removing less critical parameters to streamline the model.
Speculative Decoding: Utilizing smaller models to predict multiple tokens simultaneously, which are then verified by a larger model.
These methods are essential for optimizing LLMs while maintaining their performance.

Summary of Techniques
1. Quantization for LLMs
Reduces the numerical precision of model weights and activations.
Converts FP32 (32-bit floating point) to smaller representations like INT8 (8-bit integer) to save memory.
Balances the trade-offs between model size and language handling capabilities.
2. Pruning LLMs
Involves identifying and removing less critical parameters or entire components.
Targets redundant structures, such as attention heads or feedforward layers.
Aims to maintain the model's emergent abilities while reducing complexity.
3. Speculative Decoding for LLMs
Breaks the autoregressive generation bottleneck by predicting multiple tokens in parallel.
Uses smaller "draft" models to generate token candidates.
The larger model verifies these candidates, speeding up the generation process.
Key Takeaways
Techniques like quantization, pruning, and speculative decoding are crucial for optimizing LLMs.
These methods help create smaller, faster models without significantly sacrificing accuracy.
Understanding and applying these strategies will enhance the efficiency of LLM deployment in various applications.

# 2.

Introduction
Quantization is a technique used to make large language models (LLMs) more efficient for inference. This process reduces the numerical precision of model weights and activations, which can significantly decrease memory usage and improve performance. Understanding quantization is essential for optimizing LLMs, especially given their large size and complexity.

What is Quantization?
Quantization involves converting high-precision numbers (like 32-bit floating-point) to lower-precision formats. This is particularly important for LLMs, where model weights can be hundreds of gigabytes. Common formats include:

FP16 (16-bit floating-point): A starting point for many LLMs, offering a 2x reduction in size compared to FP32.
BFloat16: A variant of FP16 that maintains a wider dynamic range, beneficial for training and inference.
INT8 (8-bit integer): Reduces model size by another 2x compared to FP16, making it a popular choice for quantization.
INT4 (4-bit integer): Provides aggressive size reduction, potentially making models 8x smaller than FP32.
Benefits of Quantization
Quantization offers several advantages:

Reduced Memory Footprint: Lower precision means smaller weight tensors, allowing larger models to fit into available memory.
Faster Inference: Smaller data types reduce memory bandwidth usage and enable faster compute operations on GPUs.
Lower Latency and Cost: Efficient quantization can lower the cost per token when serving LLMs at scale.
Trade-offs of Quantization
While quantization has many benefits, there are trade-offs to consider:

Loss of Nuance: Aggressive compression can lead to a loss of subtle patterns in the model, affecting performance on complex tasks.
Outlier Misrepresentation: Naive quantization may misrepresent unusually large values, leading to accuracy issues. Outlier-aware techniques and mixed-precision approaches can help mitigate this.
Common Quantization Tools
Several tools facilitate the implementation of quantization:

Bitsandbytes: Allows loading models in 8-bit or 4-bit precision within the Hugging Face Transformers ecosystem.
Llama.cpp: Designed for efficient CPU inference with a focus on integer quantization.
TensorRT-LLM: Offers robust quantization capabilities for accelerating large models on NVIDIA GPUs.
Summary
Quantization is a powerful technique for optimizing large language models, making them smaller and more efficient without sacrificing accuracy. By understanding the different formats and tools available, it becomes easier to implement quantization effectively.

Key Takeaways
Quantization significantly reduces the memory footprint and improves inference speed for large language models.
Careful consideration of trade-offs is essential to maintain model performance while implementing quantization.

# 3.

Introduction
Model pruning is a technique used to reduce the size and computational cost of large language models (LLMs). By identifying and removing parts of the model that are less important, pruning aims to create a smaller, more efficient model that maintains performance close to the original. This approach is particularly useful given that LLMs are often overparameterized, meaning they have more parameters than necessary for their tasks.

What is Model Pruning?
Model pruning involves the removal of redundant or less impactful parameters from a model. This can include:

Individual weights
Entire neurons
Attention heads
Entire layers (in aggressive scenarios)
The goal is to create a smaller model that is computationally cheaper while ideally retaining similar performance levels.

Types of Pruning
Unstructured Pruning
Removes small individual weights across the model's layers.
Creates sparse weight matrices, often achieving up to 80% sparsity with minimal impact on accuracy.
Requires specialized sparse matrix multiplication kernels (e.g., cuSPARSELt from NVIDIA, support in DeepSpeed) to realize performance gains.
Recent advancements like SparseGPT and Wanda allow for one-shot unstructured pruning, minimizing or eliminating the need for retraining.
Structured Pruning
Removes entire components of the model, such as neurons or attention heads.
Results in a smaller, dense model that can be processed by standard hardware.
More aggressive than unstructured pruning, which can lead to significant accuracy loss if important components are removed.
Techniques like contribution scores or sensitivity analysis help identify less important components.
Movement pruning is a recent method that monitors parameter movement during fine-tuning to identify components for pruning.
Benefits of Pruning in LLMs
Pruning offers several advantages:

Reduced Model Size: Smaller model files are easier to store, share, and deploy, especially on resource-limited devices.
Faster Inference: Fewer computations lead to quicker model execution, particularly with structured pruning or hardware that supports sparse operations.
Lower Memory Usage: Smaller models require less memory during training and inference, beneficial for large models or edge device deployment.
Challenges of Pruning in LLMs
Despite its benefits, pruning presents challenges:

Deciding What to Prune: Identifying which weights, neurons, or attention heads to remove is complex. Traditional methods like magnitude-based pruning may not be effective for LLMs. Advanced approaches use importance scores based on activations or gradients.
Risk of Accuracy Loss: Over-pruning or removing critical components can degrade model performance, especially for LLMs with emergent abilities like complex reasoning.
Hardware and Software Support: Unstructured pruning requires specialized libraries or accelerators to benefit from sparsity. Standard hardware is often optimized for dense matrix operations.
Summary
Model pruning is a valuable technique for optimizing large language models by reducing their size and computational demands. It involves careful analysis and decision-making to ensure that performance is not sacrificed. Both unstructured and structured pruning methods have their advantages and challenges, and recent advancements are making pruning more practical for real-world applications.

Takeaways
Model pruning reduces the size and computational cost of LLMs by removing less important parameters.
Both unstructured and structured pruning methods can be effective, but they come with distinct challenges and benefits.

# 4.

## 6. Conclusion & The Real-World Picture

This demo focused purely on the 'zeroing out' mechanics within one layer. We saw how to target a layer, apply a pruning hook, and make the change permanent. 

In a real-world application, the process is much more involved:
1.  **Global Pruning:** You would prune many layers across the entire model, not just one.
2.  **Iterative Process:** Often, pruning is done iteratively—prune a little, fine-tune, prune a little more, fine-tune again.
3.  **CRITICAL STEP: Fine-Tuning:** After pruning, the model **must** be fine-tuned on a relevant dataset. This allows the remaining weights to adjust and compensate for the ones that were removed, recovering much of the lost performance.

This demo successfully highlights the first, mechanical step in that much larger journey.

### Step 5.2: Target the Layer and Check Initial Sparsity

Using our helper function, we'll grab the specific layer we want to prune. Then, we'll check its sparsity. As expected, a normal, trained layer has virtually zero sparsity.

### Step 5.3: Apply the Pruning "Mask"

This is the first key step. We use `prune.l1_unstructured` to identify the 50% of weights with the lowest magnitude (L1 norm).

**Crucially, this does *not* immediately change the weights!** Instead, PyTorch attaches a `weight_mask` and a `weight_orig` attribute to the layer. During a forward pass, the model will now use a temporary, masked version of the weights. The original weights are preserved for now.

### Step 5.4: Make the Pruning Permanent

To make our changes permanent, we call `prune.remove`. This function does two things:
1.  It removes the pruning mask and the original weight backup.
2.  It updates the layer's `weight` attribute to be the final, zeroed-out tensor.

After this step, the weights are permanently gone.



## 6. Conclusion & The Real-World Picture

This demo focused purely on the 'zeroing out' mechanics within one layer. We saw how to target a layer, apply a pruning hook, and make the change permanent. 

In a real-world application, the process is much more involved:
1.  **Global Pruning:** You would prune many layers across the entire model, not just one.
2.  **Iterative Process:** Often, pruning is done iteratively—prune a little, fine-tune, prune a little more, fine-tune again.
3.  **CRITICAL STEP: Fine-Tuning:** After pruning, the model **must** be fine-tuned on a relevant dataset. This allows the remaining weights to adjust and compensate for the ones that were removed, recovering much of the lost performance.

This demo successfully highlights the first, mechanical step in that much larger journey.

# 5.

# Demo: Speculative Decoding - A Step-by-Step Look at the Logic

**Welcome!**

In this demo, we'll pull back the curtain on **Speculative Decoding**, one of the most clever techniques for speeding up LLM inference. We won't build a full, optimized loop, but instead, we'll walk through a **single, detailed step** to understand the core logic.

**The Two Players:**
1.  **The Draft Model (`gpt2`):** A small, fast model. Think of it as a scout that runs ahead and quickly suggests a path.
2.  **The Target Model (`gpt2-medium`):** A larger, more accurate, but slower model. This is the general who verifies the scout's path.

**Our Goal:** To see how the general (target model) can efficiently verify the scout's suggestions (draft tokens) and accept multiple tokens for the cost of just one slow operation.

Takeaways
Speculative decoding can significantly reduce the number of passes required by the target model, improving overall inference speed.
The choice of K is crucial; a balance must be struck between speed and accuracy.
An optimal K value can lead to the best performance, minimizing wall-clock time while maximizing accepted tokens.
The analysis of results should focus on how varying K impacts both the efficiency and accuracy of the decoding process.

## 3. Load the Models (Our "Players")

Let's load both the small draft model and the larger target model. We'll also load the tokenizer, which is the same for both `gpt2` and `gpt2-medium`.

### Step 4.2: The General Reviews the Plan (Target Model Verification)

Now for the clever part. We take the **original context + the draft tokens** and feed this entire sequence to the large target model in a **single forward pass**.

Because Transformers process all tokens in parallel, this single pass gives us the target model's prediction for what token should come after *each* position in the input. This is far more efficient than calling the target model K times.

### Step 4.3: The Verdict (Comparison and Acceptance)

Now we compare the scout's path with the general's preferred path, one step at a time. We stop as soon as we find a mismatch.

| Position | Draft Model's Token | Target Model's Preference | Result   |
|----------|-----------------------|-----------------------------|----------|
| 1        | ...                   | ...                         | Match?   |
| 2        | ...                   | ...                         | Match?   |
| ...      | ...                   | ...                         | ...      |

In [1]:
# --- Step 3: Comparing draft against target preferences ---
# Pos 1: Draft (' is') vs Target (' is')
#   ✅ Match!
# Pos 2: Draft (' to') vs Target (' to')
#   ✅ Match!
# Pos 3: Draft (' use') vs Target (' use')
#   ✅ Match!
# Pos 4: Draft (' a') vs Target (' a')
#   ✅ Match!
# Pos 5: Draft (' simple') vs Target (' single')
#   ❌ Mismatch! Halting comparison.

# Number of matched tokens: 4


### Step 4.4: The Outcome (Constructing the Final Output)

Based on the comparison, we can now form our final output for this step. The rule is:

**`final_tokens = [all matched tokens] + [the target's 'correct' token at the mismatch point]`**

If all tokens matched, the second part is simply the next token the target model would have generated anyway.

In [ ]:
def run_speculative_decoding(draft_model, target_model, tokenizer, prompt_text, max_tokens, k):
    """Runs a speculative decoding loop for a given k and measures performance."""
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    target_passes = 0
    total_accepted_tokens = 0
    
    start_time = time.time()
    with torch.no_grad():
        while input_ids.shape[1] < (len(tokenizer.encode(prompt_text)) + max_tokens):
            # 1. Draft Phase: The small model generates K candidate tokens
            draft_ids = draft_model.generate(input_ids, max_new_tokens=k, pad_token_id=tokenizer.eos_token_id)
            draft_candidates = draft_ids[:, input_ids.shape[1]:]
            num_drafted = draft_candidates.shape[1]
            if num_drafted == 0: break # No more tokens can be drafted

            # 2. Verification Phase: The target model gets the draft + context
            verification_input = torch.cat([input_ids, draft_candidates], dim=1)
            target_logits = target_model(verification_input).logits
            target_passes += 1

            # 3. Acceptance Logic: Compare draft with target's preferences
            num_matched = 0
            for i in range(num_drafted):
                # Get the target's prediction for the i-th draft token position
                verification_logit_idx = input_ids.shape[1] + i - 1
                target_pred_id = torch.argmax(target_logits[:, verification_logit_idx, :], dim=-1)
                
                if draft_candidates[0, i] == target_pred_id.item():
                    num_matched += 1
                else:
                    break # Mismatch found, stop comparing
            
            # Accept all matched tokens
            accepted_tokens = draft_candidates[:, :num_matched]
            input_ids = torch.cat([input_ids, accepted_tokens], dim=1)
            
            # If there was a mismatch, accept the target's correction
            if num_matched < num_drafted:
                correction_logit_idx = input_ids.shape[1] -1
                correction_id = torch.argmax(target_logits[:, correction_logit_idx, :], dim=-1, keepdim=True)
                input_ids = torch.cat([input_ids, correction_id], dim=1)
            
            if tokenizer.eos_token_id in input_ids[0]: break

    end_time = time.time()
    
    total_accepted_tokens = input_ids.shape[1] - len(tokenizer.encode(prompt_text))
    avg_accepted_per_pass = total_accepted_tokens / target_passes if target_passes > 0 else 0
    
    return end_time - start_time, target_passes, avg_accepted_per_pass

--- Speculative Decoding Experiment Results Summary ---
Baseline Performance: Time=1.17s, Target Passes=60
    K  Time (s)  Target Passes  Avg. Accepted Tokens
0   1  1.861169             60              1.000000
1   2  1.211026             33              1.818182
2   3  1.060563             24              2.541667
3   4  1.047493             20              3.100000
4   5  1.021241             17              3.529412
5   8  1.186374             14              4.714286
6  10  1.204962             12              5.000000

### Guiding Questions for Analysis

1.  **Target Passes**: How did the number of expensive target model passes change as `K` increased? Was it always fewer than the baseline?
2.  **Wall-Clock Time**: How did the total generation time change with `K`? Was there an optimal `K` value that resulted in the fastest time? Why do you think time might start to increase again for very large `K`?
3.  **Average Accepted Tokens**: How did the average number of tokens accepted per verification step change with `K`? What does this metric tell you about the efficiency of the process?
4.  **Trade-offs & Conclusion**: What are the trade-offs of choosing a small `K` versus a large `K`? Based on your results, what would be the best `K` to use for this specific draft/target model pair?

### Sample Analysis

1.  **Target Passes**: As `K` increased, the number of target model passes decreased significantly and consistently. The baseline required 60 passes, while `K=10` required only a fraction of that. This is because a larger `K` allows more tokens to be verified in a single batch, reducing the number of verification steps needed to generate the full sequence.

2.  **Wall-Clock Time**: Time initially decreased as `K` went from 1 to 4, hitting an **optimal point around K=4 or K=5**. For `K` values larger than that, the total time began to increase again. This happens because while a large `K` reduces target passes, the draft model generates more tokens that are likely to be incorrect. The overhead of generating these useless draft tokens, combined with processing a larger verification batch in the target model, eventually outweighs the benefit of fewer verification steps.

3.  **Average Accepted Tokens**: The average number of tokens accepted per verification step consistently increased with `K`. This metric is a great measure of efficiency; a value greater than 1.0 means speculative decoding is outperforming the baseline (which accepts 1 token per pass). A higher number indicates the draft model's predictions are often correct, allowing us to accept multiple tokens for the cost of one target pass.

4.  **Trade-offs & Conclusion**:
    *   **Small `K` (e.g., 1-2)**: Safe and low overhead. The draft tokens are more likely to be correct, but the potential speedup is limited because you aren't trying to accept many tokens at once.
    *   **Large `K` (e.g., 8-10)**: High risk, high reward. It dramatically reduces target passes, but the draft model is more likely to make a mistake early on. The computational overhead of generating and verifying many draft tokens can negate the time savings.
    *   **Optimal `K` (e.g., 4-5)**: This is the "sweet spot." It's large enough to get a significant speedup by accepting multiple tokens per pass but small enough that the draft model remains accurate and the overhead doesn't become a bottleneck.

For this `gpt2`/`gpt2-medium` pairing, a **`K` value of 4 or 5 is the optimal choice**, providing the best wall-clock time speedup.

es how speculative decoding allows for the generation of multiple tokens while incurring the computational cost of the target model only once. This method can lead to significant speed improvements without sacrificing output quality.

Takeaways
Speculative decoding combines the strengths of a fast draft model and an accurate target model.
The process reduces the number of expensive calls to the target model, enhancing efficiency.
The approach maintains output quality by allowing the target model to verify and correct draft tokens.
This technique is beneficial in applications requiring rapid text generation.